1. instalasi dan inport libary

In [17]:
import requests
import folium
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from shapely.geometry import shape
from pyproj import CRS
import json
import os

2. Mengambil data dari Mapid

In [18]:
urls = {
    "tingkat_polusi": "https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=6984a42396de9425652debb4&project_id=697e526a949d92a51f0b4816",
    "kepadatan_penduduk": "https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=6984acd196de9425652f4015&project_id=697e526a949d92a51f0b4816"
}

In [19]:
def get_gdf_from_api(url):
    response = requests.get(url)
    if response.status_code == 200:
        geojson = response.json()
        gdf = gpd.GeoDataFrame.from_features(geojson['features'])
        if gdf.crs is None:
            gdf.set_crs(epsg=4326, inplace=True)
        return gdf
    else:
        print(f"Gagal mengambil data dari {url}") [cite: 277]
        return gpd.GeoDataFrame()

In [20]:
gdfs = {}
for key, url in urls.items():
    gdfs[key] = get_gdf_from_api(url)

print("Data Polusi dan Kepadatan Penduduk Berhasil Dimuat.")

Data Polusi dan Kepadatan Penduduk Berhasil Dimuat.


3. Simulasi Data untuk mendapatkan data Polusi Suara

In [21]:
gdf_polusi = gdfs['tingkat_polusi'].copy()

np.random.seed(42)
gdf_polusi['POLUSI_SUARA'] = np.random.randint(4, 9, size=len(gdf_polusi))

def kategori_suara(val):
    if val >= 8: return "TINGGI"
    elif val >= 6: return "SEDANG"
    else: return "RENDAH"

gdf_polusi['KAT_SUARA'] = gdf_polusi['POLUSI_SUARA'].apply(kategori_suara)
gdfs['tingkat_polusi'] = gdf_polusi

print(gdf_polusi[['geometry', 'TINGKAT_POLUSI', 'KAT_SUARA']].head())

                                            geometry TINGKAT_POLUSI KAT_SUARA
0  POLYGON ((106.88 -6.28, 106.89 -6.28, 106.89 -...         RENDAH    SEDANG
1  POLYGON ((106.86 -6.26, 106.87 -6.26, 106.87 -...         SEDANG    TINGGI
2  POLYGON ((106.84 -6.24, 106.85 -6.24, 106.85 -...         RENDAH    SEDANG
3  POLYGON ((106.82 -6.22, 106.83 -6.22, 106.83 -...         TINGGI    TINGGI
4  POLYGON ((106.8 -6.2, 106.81 -6.2, 106.81 -6.2...         SEDANG    TINGGI


Cek data null dan CRS

In [23]:
for key, gdf in gdfs.items():
    for col in gdf.columns:
        if gdf[col].dtype == 'object' and col != 'geometry':
            gdf[col] = gdf[col].fillna('tidak diketahui')
        else:
            gdf[col] = gdf[col].fillna(0)

if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326, inplace=False)
    gdfs[key] = gdf

print("Data sudah dibersihkan dan siap untuk analisis lebih lanjut.")

Data sudah dibersihkan dan siap untuk analisis lebih lanjut.


In [24]:
for key, gdf in gdfs.items():
    m = folium.Map(location=[-6.200000, 106.816666], zoom_start=11)
    folium.GeoJson(gdf).add_to(m)
    folium.LayerControl().add_to(m)
    display(m)
    print(f"Layer '{key}' telah ditampilkan di peta interaktif.")

Layer 'tingkat_polusi' telah ditampilkan di peta interaktif.


Layer 'kepadatan_penduduk' telah ditampilkan di peta interaktif.


4. Data Lapangan Padel di Jakarta

In [15]:
padel_data = [
    {"name": "Padel Pro Jakarta", "lat": -6.2625, "lon": 106.8112, "zona": "Komersial"},
    {"name": "Fitz Padel SCBD", "lat": -6.2244, "lon": 106.8098, "zona": "Campuran"},
    {"name": "Padel Court Kelapa Gading", "lat": -6.1622, "lon": 106.9011, "zona": "Permukiman"},
    {"name": "Z Padel Indonesia", "lat": -6.2120, "lon": 106.8230, "zona": "Komersial"},
    {"name": "Padel Kemang Cluster", "lat": -6.2730, "lon": 106.8200, "zona": "Permukiman"}
]

In [16]:
nodes = [Point(p["lon"], p["lat"]) for p in padel_data]
gdf_padel = gpd.GeoDataFrame(padel_data, geometry=nodes, crs="EPSG:4326")

print("Data Lapangan Padel (Hasil Scraping) Berhasil Dikonversi ke Spasial:")
print(gdf_padel[['name', 'geometry']].head())

Data Lapangan Padel (Hasil Scraping) Berhasil Dikonversi ke Spasial:
                        name                  geometry
0          Padel Pro Jakarta  POINT (106.8112 -6.2625)
1            Fitz Padel SCBD  POINT (106.8098 -6.2244)
2  Padel Court Kelapa Gading  POINT (106.9011 -6.1622)
3          Z Padel Indonesia    POINT (106.823 -6.212)
4       Padel Kemang Cluster     POINT (106.82 -6.273)
